In [1]:
import os
import re
import sys
from pathlib import Path
from urllib.parse import quote

os.environ.setdefault("JAX_PLATFORMS", "cpu")

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

import pandas as pd
import requests

repo_root = Path.cwd()
if not (repo_root / "braincell").exists():
    if (repo_root.parent / "braincell").exists():
        repo_root = repo_root.parent
    else:
        raise RuntimeError("Run this notebook from the repository root or develop_doc/.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import braincell
from braincell import Morpho
from develop_doc.helper import (
    dict_to_df,
    morpho_report_df,
    morpho_report_summary_df,
    morpho_summary_df,
)

print("braincell version:", braincell.__version__)
print("repo_root:", repo_root)


/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.1 is installed, but it is not compatible with the installed jaxlib version 0.6.2, so it will not be used.
  warnings.warn(


braincell version: 0.0.8
repo_root: /home/swl/braincell


# `io`: query NeuroMorpho metadata, download `SWC`, then load with `braincell`

This notebook demonstrates the smallest practical NeuroMorpho workflow in code:

1. Query the public REST API with a simple filter
2. Preview the matching neuron records
3. Download one standardized `SWC` file
4. Import the downloaded file with `Morpho.from_swc(...)`

The example keeps the filter intentionally small: one main query field plus one extra facet filter.


## 1. Query NeuroMorpho with `q` and `fq`

NeuroMorpho exposes neuron metadata from `https://neuromorpho.org/api/neuron/select/`.

In practice, the easiest pattern is:

- `q="species:mouse"` for the main query
- `fq=["brain_region:neocortex"]` for extra filters

You can discover supported field names from `https://neuromorpho.org/api/neuron/fields`.


In [2]:
API_BASE = "https://neuromorpho.org/api"
FILE_BASE = "https://neuromorpho.org/dableFiles"
session = requests.Session()


def fetch_neurons(*, query, filters=None, size=5, page=0, sort="neuron_id,asc"):
    params = {
        "q": query,
        "size": size,
        "page": page,
        "sort": sort,
    }
    if filters:
        params["fq"] = [f"{key}:{value}" for key, value in filters.items()]

    response = session.get(f"{API_BASE}/neuron/select/", params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    neurons = payload.get("_embedded", {}).get("neuronResources", [])
    return neurons, payload.get("page", {}), response.url


def _as_text(value):
    if isinstance(value, list):
        return "; ".join(str(x) for x in value)
    return value


def neurons_to_df(neurons):
    rows = []
    for neuron in neurons:
        rows.append(
            {
                "neuron_id": neuron["neuron_id"],
                "neuron_name": neuron["neuron_name"],
                "archive": neuron["archive"],
                "species": neuron["species"],
                "brain_region": _as_text(neuron.get("brain_region")),
                "cell_type": _as_text(neuron.get("cell_type")),
                "experiment_condition": _as_text(neuron.get("experiment_condition")),
            }
        )
    return pd.DataFrame(rows)


def standardized_swc_url(neuron):
    archive = quote(neuron["archive"].lower(), safe="")
    neuron_name = quote(neuron["neuron_name"], safe="")
    return f"{FILE_BASE}/{archive}/CNG%20version/{neuron_name}.CNG.swc"


def safe_filename(name):
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    return cleaned.strip("._") or "neuromorpho_neuron"


def download_swc(neuron, output_dir, *, overwrite=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    filename = f"{safe_filename(neuron['neuron_name'])}.CNG.swc"
    output_path = output_dir / filename
    if output_path.exists() and not overwrite:
        return output_path, standardized_swc_url(neuron), False

    url = standardized_swc_url(neuron)
    with session.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with output_path.open("wb") as fh:
            for chunk in response.iter_content(chunk_size=1 << 15):
                if chunk:
                    fh.write(chunk)
    return output_path, url, True


In [3]:
query = "species:mouse"
filters = {"brain_region": "neocortex"}

neurons, page_info, query_url = fetch_neurons(
    query=query,
    filters=filters,
    size=5,
)
if not neurons:
    raise RuntimeError("No NeuroMorpho records matched the current filters. Relax the filters and run again.")

display(
    dict_to_df(
        {
            "query": query,
            "filters": filters,
            "request_url": query_url,
            "total_matches": page_info.get("totalElements"),
            "page_size": page_info.get("size"),
        }
    )
)
display(neurons_to_df(neurons))

selected_neuron = neurons[0]
display(
    dict_to_df(
        {
            "selected_neuron": selected_neuron["neuron_name"],
            "archive": selected_neuron["archive"],
            "download_url": standardized_swc_url(selected_neuron),
        }
    )
)


,key,value
0,query,species:mouse
1,filters,{'brain_region': 'neocortex'}
2,request_url,https://neuromorpho.org/api/neuron/select/?q=s...
3,total_matches,54580
4,page_size,5


,neuron_id,neuron_name,archive,species,brain_region,cell_type,experiment_condition
0,10047,TypeA-10,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control
1,10048,TypeB-06,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control
2,10049,ChR2-negative-03,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control
3,10050,ChR2-negative-06,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control
4,10051,TypeA-09,Scanziani,mouse,neocortex; occipital; layer 6,principal cell,Control


,key,value
0,selected_neuron,TypeA-10
1,archive,Scanziani
2,download_url,https://neuromorpho.org/dableFiles/scanziani/C...


## 2. Download one standardized `SWC`

NeuroMorpho exposes normalized files under a predictable URL pattern:

- `https://neuromorpho.org/dableFiles/{archive}/CNG version/{neuron_name}.CNG.swc`

The helper above builds this URL from the selected metadata row and saves the file locally.


In [4]:
output_dir = repo_root / "develop_doc" / "data" / "neuromorpho"
downloaded_path, download_url, downloaded_now = download_swc(
    selected_neuron,
    output_dir,
    overwrite=False,
)

display(
    dict_to_df(
        {
            "download_url": download_url,
            "saved_to": downloaded_path,
            "downloaded_now": downloaded_now,
            "file_size_bytes": downloaded_path.stat().st_size,
        }
    )
)

header_preview = downloaded_path.read_text(errors="replace").splitlines()[:5]
print("first lines:")
print("\n".join(header_preview))


,key,value
0,download_url,https://neuromorpho.org/dableFiles/scanziani/C...
1,saved_to,/home/swl/braincell/develop_doc/data/neuromorp...
2,downloaded_now,True
3,file_size_bytes,80865


first lines:
# Original file TypeA-10.swc edited using StdSwc version 1.31 on 5/5/14.
# Irregularities and fixes documented in TypeA-10.swc.std.  See StdSwc1.31.doc for more information.
#
# Neurolucida to SWC conversion from L-Measure. Sridevi Polavaram: spolavar@gmu.edu
# Original fileName:C:\Users\praveen\Desktop\DataProcessing\CurrentArchives\Processing\Olsen-Scanziani\ASC\TypeA-10.asc


## 3. Load the downloaded file with `braincell`

Once the `SWC` file is on disk, the rest of the workflow is the same as any local import:

- `tree = Morpho.from_swc(path)`
- `tree, report = Morpho.from_swc(path, return_report=True)`


In [5]:
tree, report = Morpho.from_swc(downloaded_path, return_report=True)

display(morpho_summary_df(tree))
display(morpho_report_summary_df(report))
display(morpho_report_df(report).head(10))
print("topology preview:")
print("\n".join(tree.topo().splitlines()[:8]))


,key,value
0,root_name,soma
1,root_type,soma
2,n_branches,22
3,n_stems,1
4,n_bifurcations,10
5,max_branch_order,9
6,total_length,1402.9893 * umetre
7,total_area,607.0726 * umeter2
8,total_volume,208.99648 * flitre
9,mean_radius,0.06886624 * umetre


,key,value
0,n_issues,2
1,n_errors,0
2,n_warnings,2
3,has_errors,False
4,has_warnings,True
5,issue_codes,"(identity.sequential_index, topology.sorted_or..."


,severity,code,message,line_number,node_id,fix_message,fix_applied
0,warning,topology.sorted_order,SWC rows were reordered so parents appear befo...,None,None,sort rows into parent-before-child order,True
1,warning,identity.sequential_index,SWC node ids were renumbered to a sequential 1...,None,None,renumber indices sequentially,True


topology preview:
soma
└── apical_dendrite_0
    ├── apical_dendrite_1
    └── apical_dendrite_2
        ├── apical_dendrite_3
        │   ├── apical_dendrite_4
        │   │   ├── apical_dendrite_5
        │   │   │   ├── apical_dendrite_6


## 4. Next adjustments

To adapt this example for your own download script, the main knobs are:

- change `query`, for example `"species:rat"`
- add or remove entries in `filters`, for example `{"brain_region": "hippocampus"}`
- increase `size` if you want to preview more matches before choosing one
- set `overwrite=True` in `download_swc(...)` if you want to refresh a local copy

If you need the full list of filterable fields, call `GET https://neuromorpho.org/api/neuron/fields` and reuse the same `field:value` syntax shown here.
